# Essentia Mood Models — Valence Key Correction

Same benchmark as the cross-model comparison notebook (`01b`), but with the
key-mode valence correction from `feat/valence_key_correction` applied on top
of every model's raw output.

**Correction formula:** `valence += α × (is_major − 0.5) × key_strength`  
where `α = 0.13` (calibrated on EmoMusic: mean valence major ≈ 5.8/9 vs minor ≈ 4.6/9)
and `key_strength` is the HPCP-based KeyExtractor confidence ∈ [0, 1].

| Model tag | Backbone | Head trained on | Reported R² (V / A) |
|-----------|----------|-----------------|---------------------|
| `emomusic-musicnn` | MusiCNN | EmoMusic (744 songs) | 0.515 / 0.646 |
| `deam-musicnn` | MusiCNN | DEAM (1802 songs) | 0.537 / 0.589 |
| `emomusic-vggish` | VGGish | EmoMusic (744 songs) | 0.461 / 0.612 |
| `deam-vggish` | VGGish | DEAM (1802 songs) | 0.524 / 0.591 |

> **NOTE:** essentia-tensorflow is CPU-only — selecting a GPU runtime does not help here.

### Sections
0. Environment setup
1. Shared evaluation utilities
2. Download model files
3. Parameterised predictor (with key-mode valence correction)
4. Datasets
5. Evaluation (all models × all datasets)
6. Cross-model comparison
7. Save results


## 0. Environment Setup

In [ ]:
!pip install essentia-tensorflow yt-dlp librosa matplotlib pandas scipy tqdm gdown -q
print("Setup complete.")

## 1. Shared Evaluation Utilities

In [ ]:
import os, sys
from pathlib import Path

# REPO_BRANCH: update to "main" after the PR is merged
REPO_BRANCH = "feat/mood-model-benchmark"
REPO_NAME   = "Soundtrack-Mood-Manager"

_cwd = Path.cwd()
if (_cwd / "eval_datasets.py").exists():
    _eval_dir = _cwd
else:
    _repo_root = _cwd / REPO_NAME
    if not (_repo_root / "evaluation").exists():
        !git clone --depth 1 --branch {REPO_BRANCH} \
            https://github.com/francescovidaich964/{REPO_NAME}.git
    _eval_dir = _repo_root / "evaluation"

_eval_dir = _eval_dir.resolve()
if str(_eval_dir) not in sys.path:
    sys.path.insert(0, str(_eval_dir))

from eval_datasets import setup_deam, setup_emomusic, setup_pmemo, setup_merge
from metrics import compute_metrics, print_metrics
from visualization import plot_scatter, cross_dataset_comparison
from spot_checks import SPOT_CHECKS, download_spot_checks, run_evaluation, profile_predictor

import math, warnings, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR      = Path("data")
MODELS_DIR    = Path("models")
SPOTCHECK_DIR = Path("spotchecks")
for d in [DATA_DIR, MODELS_DIR, SPOTCHECK_DIR]:
    d.mkdir(exist_ok=True)

print(f"Imports complete. (eval_dir={_eval_dir})")

## 2. Download Model Files

Two feature extractors (MusiCNN, VGGish) and four regression heads.

In [ ]:
MODEL_URLS = {
    # Feature extractors
    "msd-musicnn-1.pb":          "https://essentia.upf.edu/models/feature-extractors/musicnn/msd-musicnn-1.pb",
    "audioset-vggish-3.pb":      "https://essentia.upf.edu/models/feature-extractors/vggish/audioset-vggish-3.pb",
    # Regression heads
    "emomusic-msd-musicnn-2.pb": "https://essentia.upf.edu/models/classification-heads/emomusic/emomusic-msd-musicnn-2.pb",
    "deam-msd-musicnn-2.pb":     "https://essentia.upf.edu/models/classification-heads/deam/deam-msd-musicnn-2.pb",
    "emomusic-audioset-vggish-2.pb": "https://essentia.upf.edu/models/classification-heads/emomusic/emomusic-audioset-vggish-2.pb",
    "deam-audioset-vggish-2.pb":     "https://essentia.upf.edu/models/classification-heads/deam/deam-audioset-vggish-2.pb",
}

for name, url in MODEL_URLS.items():
    dest = MODELS_DIR / name
    if dest.exists():
        print(f"  ✓ {name} ({dest.stat().st_size/1e6:.1f} MB, cached)")
        continue
    print(f"Downloading {name}...", end=" ", flush=True)
    r = requests.get(url, stream=True, timeout=120)
    r.raise_for_status()
    with open(dest, "wb") as f:
        for chunk in r.iter_content(65536):
            f.write(chunk)
    print(f"{dest.stat().st_size/1e6:.1f} MB")

## 3. Parameterised Predictor

In [ ]:
# Valence correction alpha: calibrated on EmoMusic (major vs minor gap ≈ 0.13)
_ALPHA = 0.13


class EssentiaPredictor:
    """Essentia valence/arousal predictor with key-mode valence correction.

    Identical to the baseline predictor except for the extra KeyExtractor step
    that shifts valence up for major-key tracks and down for minor-key tracks,
    weighted by KeyExtractor confidence so ambiguous/atonal tracks get ≈ zero
    correction.  Full-confidence major: +0.065 | full-confidence minor: −0.065.
    """

    _EXTRACTOR_CFG = {
        "musicnn": ("TensorflowPredictMusiCNN", "model/dense/BiasAdd"),
        "vggish":  ("TensorflowPredictVGGish",  "model/vggish/embeddings"),
    }

    def __init__(self, models_dir: Path, extractor_pb: str, head_pb: str,
                 batch_size: int = 256):
        import essentia
        essentia.log.warningActive = False

        backbone = "musicnn" if "musicnn" in extractor_pb else "vggish"
        cls_name, extractor_output = self._EXTRACTOR_CFG[backbone]

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            from essentia.standard import MonoLoader, TensorflowPredict2D, KeyExtractor
            import essentia.standard as _es
            ExtractorCls = getattr(_es, cls_name)

        self._MonoLoader = MonoLoader
        self._extractor = ExtractorCls(
            graphFilename=str(models_dir / extractor_pb),
            output=extractor_output,
            batchSize=batch_size,
        )
        self._head = TensorflowPredict2D(
            graphFilename=str(models_dir / head_pb),
            output="model/Identity",
            batchSize=batch_size,
        )
        self._key_extractor = KeyExtractor(sampleRate=16000)
        print(f"EssentiaPredictor (valence-corrected) ready: {extractor_pb} → {head_pb}")

    def predict(self, audio_path) -> dict | None:
        """Returns {'valence': float, 'arousal': float} in [0, 1], or None on failure."""
        try:
            audio      = self._MonoLoader(filename=str(audio_path), sampleRate=16000)()

            # Key detection for valence correction.
            try:
                _key, scale, key_strength = self._key_extractor(audio)
                is_major = 1.0 if scale == "major" else 0.0
            except Exception:
                is_major, key_strength = 0.5, 0.0  # neutral: no correction

            embeddings = self._extractor(audio)
            if embeddings.shape[0] == 0:
                return None
            preds = self._head(embeddings)
            if preds.shape[0] == 0:
                return None
            mean    = preds.mean(axis=0)
            valence = float(np.clip((mean[0] - 1.0) / 8.0, 0.0, 1.0))
            arousal = float(np.clip((mean[1] - 1.0) / 8.0, 0.0, 1.0))

            # Apply key-mode valence correction weighted by detection confidence.
            valence = float(np.clip(valence + _ALPHA * (is_major - 0.5) * key_strength, 0.0, 1.0))

            if not (math.isfinite(valence) and math.isfinite(arousal)):
                return None
            return {"valence": valence, "arousal": arousal}
        except Exception:
            return None


# Models to compare: tag -> (extractor_pb, head_pb)
ESSENTIA_MODELS = {
    "emomusic-musicnn": ("msd-musicnn-1.pb",     "emomusic-msd-musicnn-2.pb"),
    "deam-musicnn":     ("msd-musicnn-1.pb",     "deam-msd-musicnn-2.pb"),
    "emomusic-vggish":  ("audioset-vggish-3.pb", "emomusic-audioset-vggish-2.pb"),
    "deam-vggish":      ("audioset-vggish-3.pb", "deam-audioset-vggish-2.pb"),
}

print(f"Models to benchmark: {list(ESSENTIA_MODELS)}")


## 4. Datasets

Annotations downloaded automatically where possible. Audio for DEAM and MERGE is ~2.5 GB total.

In [ ]:
df_deam,     deam_id, deam_val, deam_aro = setup_deam(DATA_DIR,  download_audio=True)
df_emomusic, em_id,   em_val,   em_aro   = setup_emomusic(DATA_DIR)
df_pmemo,    pm_id,   pm_val,   pm_aro   = setup_pmemo(DATA_DIR)
df_merge,    mg_id,   mg_val,   mg_aro   = setup_merge(DATA_DIR, download_audio=True)

In [ ]:
_deam_audio   = next(iter(sorted((DATA_DIR / "deam").rglob("MEMD_audio"))), None)
DEAM_AUDIO    = _deam_audio if _deam_audio else DATA_DIR / "deam" / "MEMD_audio"

EMOMUSIC_AUDIO = DATA_DIR / "emomusic" / "clips"

_pmemo_chorus = next(iter(sorted((DATA_DIR / "pmemo").rglob("chorus"))), None)
PMEMO_AUDIO   = _pmemo_chorus if _pmemo_chorus else DATA_DIR / "pmemo" / "chorus"

_merge_root   = next(iter(sorted((DATA_DIR / "merge").rglob("MERGE_Audio_Balanced"))), None)
MERGE_AUDIO   = _merge_root if _merge_root else DATA_DIR / "merge" / "MERGE_Audio_Balanced"

for name, path in [("DEAM", DEAM_AUDIO), ("EmoMusic", EMOMUSIC_AUDIO),
                   ("PMEmo", PMEMO_AUDIO), ("MERGE", MERGE_AUDIO)]:
    status = "✓" if path.exists() else "✗ not found"
    print(f"  {status}  {name}: {path}")

DATASETS = [
    ("DEAM",     df_deam,     DEAM_AUDIO,     deam_id, deam_val, deam_aro),
    ("EmoMusic", df_emomusic, EMOMUSIC_AUDIO, em_id,   em_val,   em_aro),
    ("PMEmo",    df_pmemo,    PMEMO_AUDIO,    pm_id,   pm_val,   pm_aro),
    ("MERGE",    df_merge,    MERGE_AUDIO,    mg_id,   mg_val,   mg_aro),
]

# set to an integer (e.g. 200) for a quick sanity check
MAX_TRACKS = None

## 5. Evaluation

Runs all 4 models on every available dataset. At ~4 s/track on CPU:
~2 h per model × 4 models ≈ **~8 h total** for DEAM + PMEmo + MERGE.
Set `MAX_TRACKS` above to a small number first to verify everything works.

In [ ]:
all_results: dict[str, pd.DataFrame] = {}  # key: "<model_tag>/<dataset>"

for model_tag, (extractor_pb, head_pb) in ESSENTIA_MODELS.items():
    predictor = EssentiaPredictor(MODELS_DIR, extractor_pb, head_pb)

    for ds_name, df_a, audio_dir, id_col, val_col, aro_col in DATASETS:
        if df_a is None or not audio_dir.exists():
            print(f"  Skipping {model_tag}/{ds_name} — audio not available")
            continue
        key = f"{model_tag}/{ds_name}"
        all_results[key] = run_evaluation(
            ds_name, model_tag, predictor.predict,
            audio_dir, df_a, id_col, val_col, aro_col, MAX_TRACKS,
        )

print(f"\nEvaluation complete. {len(all_results)} result sets.")

## 6. Cross-Model Comparison

In [ ]:
if not all_results:
    print("No results yet — run Section 5 first.")
else:
    # Print metrics table per dataset × model
    combined = pd.concat(all_results.values())

    for ds_name in combined["dataset"].unique():
        print(f"\n{'═'*60}")
        print(f"  {ds_name}")
        print(f"{'═'*60}")
        for model_tag in combined["model"].unique():
            sub = combined[(combined["dataset"] == ds_name) &
                           (combined["model"]   == model_tag)]
            if sub.empty:
                continue
            rows = []
            for dim in ["valence", "arousal"]:
                m = compute_metrics(sub, dim)
                rows.append({"Dim": dim, "n": m["n"],
                             "MAE": f"{m['mae']:.4f}", "R²": f"{m['r2']:.4f}",
                             "Pearson r": f"{m['pearson_r']:.4f}",
                             "Kendall τ": f"{m['kendall_tau']:.4f}"})
            print(f"  {model_tag}:")
            for line in pd.DataFrame(rows).to_string(index=False).split("\n"):
                print(f"    {line}")

In [ ]:
if not all_results:
    print("No results to plot.")
else:
    combined = pd.concat(all_results.values())
    cross_dataset_comparison(combined)

## 7. Save Results

In [ ]:
if all_results:
    combined = pd.concat(all_results.values())
    out = "essentia_valence_key_correction_results.csv"
    combined.to_csv(out, index=False)
    print(f"Saved: {out}  ({len(combined)} rows, {combined['model'].nunique()} models, "
          f"{combined['dataset'].nunique()} datasets)")
else:
    print("No results to save — run Section 5 first.")